# Approach E1 – KimiaNet feature extraction from slide mosaics

**Extraction of 1024-dimensional histopathology embeddings from the slide mosaics with KimiaNet (DenseNet-121)**

*Input: the mosaics produced by `01_mosaic_building.ipynb`.*

## Overview

Every 4000 × 4000 px mosaic is passed once through KimiaNet, a DenseNet-121 network trained on TCGA diagnostic slides (Riasatian et al., *Medical Image Analysis*, 2021). The output of the third-to-last layer of the network (`layers[-3]`) is averaged spatially (global average pooling) to obtain a 1024-dimensional descriptor per slide. Images are standardised with the ImageNet mean and standard deviation. One feature table is produced for each of the distances used to build the mosaics.

The KimiaNet weights (`KimiaNetKerasWeights.h5`) are distributed by the authors of KimiaNet and are **not** included in this repository.

## Contents

1. Configuration
2. Model
3. Feature extraction
4. Verification

## 1. Configuration

By default the network is executed on the CPU (`USE_GPU = False`), because a full 4000 × 4000 px mosaic requires a large amount of memory. Set `USE_GPU = True` if a GPU with sufficient memory and a working CUDA installation are available.

In [ ]:
import os

USE_GPU = False
if not USE_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""          # must be set before TensorFlow is imported
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")   # hide TensorFlow informational messages

import gc
from pathlib import Path

import numpy as np
import pandas as pd
import skimage.io as io
import tensorflow as tf

In [ ]:
DATA_DIR = Path("data")     # adjust to the location of the input files

MOSAICS_DIR = DATA_DIR / "mosaics"                                    # <distance>/<slide_id>.png
WEIGHTS_FILE = DATA_DIR / "KimiaNet" / "KimiaNetKerasWeights.h5"      # not tracked

MOSAIC_SIZE = 4000                     # px (height = width)
N_FEATURES = 1024
FEATURES_FILENAME = "KimiaNet_features.csv"
OVERWRITE = False                      # False: distances whose feature table already exists are skipped

# ImageNet statistics used to standardise the input
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

print("TensorFlow", tf.__version__, "| GPUs visible:", len(tf.config.list_physical_devices("GPU")))

## 2. Model

KimiaNet is loaded as a Keras DenseNet-121 without the classification head. Following the reference implementation, the feature map of `layers[-3]` is pooled with a global average pooling layer.

In [ ]:
if not WEIGHTS_FILE.exists():
    raise FileNotFoundError(
        f"KimiaNet weights not found at {WEIGHTS_FILE}. Download 'KimiaNetKerasWeights.h5' from the "
        "KimiaNet authors and place it there (or edit WEIGHTS_FILE)."
    )

backbone = tf.keras.applications.DenseNet121(
    include_top=False,
    weights=str(WEIGHTS_FILE),
    input_shape=(MOSAIC_SIZE, MOSAIC_SIZE, 3),
    pooling="avg",
)
feature_extractor = tf.keras.Model(
    inputs=backbone.input,
    outputs=tf.keras.layers.GlobalAveragePooling2D()(backbone.layers[-3].output),
)
print("Output shape:", feature_extractor.output_shape)

In [ ]:
def extract_features(image_path):
    """Return the KimiaNet descriptor (1-D array of length N_FEATURES) of one mosaic image."""
    mosaic = io.imread(image_path)
    if mosaic.ndim == 2:                               # grayscale -> RGB
        mosaic = np.stack([mosaic] * 3, axis=-1)
    if mosaic.shape[2] == 4:                           # drop the alpha channel
        mosaic = mosaic[:, :, :3]

    image = mosaic.astype(np.float32) / 255.0
    del mosaic
    image = (image - IMAGENET_MEAN) / IMAGENET_STD
    features = feature_extractor.predict(image[np.newaxis, ...], verbose=0)
    del image
    gc.collect()
    return features.flatten()

## 3. Feature extraction

For each sub-directory of `MOSAICS_DIR` (one per distance), the descriptors of all mosaics are stored in `<distance>/KimiaNet_features.csv`, with columns `0`, …, `1023` and `ID` (the slide identifier). Mosaics that fail are reported and omitted from the table.

In [ ]:
distance_dirs = sorted(p for p in MOSAICS_DIR.iterdir() if p.is_dir() and "." not in p.name)
print("Distances found:", [p.name for p in distance_dirs])

failures = []
for i, distance_dir in enumerate(distance_dirs, 1):
    output_csv = distance_dir / FEATURES_FILENAME
    slide_ids = sorted(f.stem for f in distance_dir.glob("*.png"))
    if output_csv.exists() and not OVERWRITE:
        print(f"[{i}/{len(distance_dirs)}] {distance_dir.name}: table already exists, skipped")
        continue

    print(f"[{i}/{len(distance_dirs)}] {distance_dir.name}: {len(slide_ids)} mosaics")
    rows, kept_ids = [], []
    for j, slide_id in enumerate(slide_ids, 1):
        try:
            rows.append(extract_features(distance_dir / f"{slide_id}.png")[:N_FEATURES])
            kept_ids.append(slide_id)
        except Exception as exc:
            failures.append((distance_dir.name, slide_id, f"{type(exc).__name__}: {exc}"))
        print(f"  {j}/{len(slide_ids)}", end="\r")

    if rows:
        table = pd.DataFrame(np.vstack(rows), columns=[str(k) for k in range(N_FEATURES)])
        table["ID"] = kept_ids
        table.to_csv(output_csv, index=False)
        print(f"  saved: {output_csv.name} ({len(table)} slides)")

print(f"\nFailures: {len(failures)}")
for failure in failures:
    print("  ", failure)

## 4. Verification

Each feature table must contain `N_FEATURES` numeric columns plus the `ID` column, and the same slides for every distance.

In [ ]:
summary = []
for distance_dir in distance_dirs:
    csv_path = distance_dir / FEATURES_FILENAME
    if csv_path.exists():
        table = pd.read_csv(csv_path)
        summary.append({"distance": distance_dir.name, "slides": len(table),
                        "feature_columns": table.shape[1] - 1,
                        "columns_ok": table.shape[1] == N_FEATURES + 1,
                        "missing_values": int(table.isna().sum().sum())})
    else:
        summary.append({"distance": distance_dir.name, "slides": 0, "feature_columns": 0,
                        "columns_ok": False, "missing_values": 0})
pd.DataFrame(summary).set_index("distance")